In [1]:
# packages and working directory  
import scipy 
import sklearn

import econml 
import arch
import os 
import pandas as pd 
import numpy as np 
import seaborn as sns 
import matplotlib as plt
import matplotlib.pyplot as plt 
import statsmodels.api as sm 
from statsmodels.discrete.discrete_model import Probit
from statsmodels.iolib.summary2 import summary_col
import statsmodels.formula.api as smf 
from scipy.optimize import minimize,fsolve, approx_fprime
from scipy.special import logsumexp
from scipy import stats
from scipy.stats import ttest_ind
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
import biogeme.database as db
import biogeme.biogeme as bio 
import biogeme.models as models 
import biogeme.version as ver 
from biogeme.expressions import Beta, log,Variable
from ipywidgets import interactive, FloatSlider
import ipywidgets as widgets
from mpl_toolkits.mplot3d import Axes3D
import pandas as pd
from statsmodels.stats.outliers_influence import variance_inflation_factor
from line_profiler import LineProfiler
# Adjust display settings to prevent truncation
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

# Consolidate changing directory and CPI dictionary since these don't change throughout the script
new_directory = r'C:\Users\hisham\Spain\2021 datasets'
os.chdir(new_directory)


In [2]:
# Read the data files and data prep 
df_h0 = pd.read_csv('H0.txt', sep='\t')
df_h1 = pd.read_csv('H1.txt', sep='\t')
df_h2 = pd.read_csv('H2.txt', sep='\t')
df_h3 = pd.read_csv('H3.txt', sep='\t')
filtered_df = pd.read_csv('1hhdmodel.csv')

# Assign scenario identifiers
df_h0['scenario'] = 'h0'
df_h1['scenario'] = 'h1'
df_h2['scenario'] = 'h2'
df_h3['scenario'] = 'h3'

# Concatenate into long format DataFrame
long_df = pd.concat([df_h0, df_h1, df_h2, df_h3], ignore_index=True)

# Merge with filtered_df to include yds_other and other necessary columns
long_df = long_df.merge(filtered_df[['idperson','original_scenario', 'lhw_h0', 'lhw_h1', 'lhw_h2', 'lhw_h3', 'yds_other']], on='idperson', how='left')

# Create 'choice_made' binary variable: 1 if the scenario matches the original choice, 0 otherwise
long_df['choice_made'] = (long_df['scenario'] == long_df['original_scenario']).astype(int)

# Calculate leisure time
long_df['leisure'] = 80 - long_df['lhw']

# Sort by 'idperson'
long_df = long_df.sort_values(by='idperson')

#new household income 
long_df['hyds'] = long_df['yds_other'] + long_df['ils_udb_yds'] 
# Optionally, save the long format DataFrame to a file
long_df.to_csv('long_format_dataset.csv', index=False)


In [3]:
# Assuming long_df is your original DataFrame
mode_deh = long_df['deh'].mode()[0]
mode_dms = long_df['dms'].mode()[0]
mode_dcz = long_df['dcz'].mode()[0]
mode_lindi = long_df['lindi'].mode()[0]
mode_amrtn = long_df['amrtn'].mode()[0]
mode_les = long_df['les'].mode()[0]



In [4]:
print(mode_amrtn)
print(mode_dcz)
print(mode_deh)
print(mode_dms)
print(mode_lindi)
print(mode_les)

1
1
5
1
2
3


In [5]:
long_df['lindi'].value_counts()

lindi
2     3156
4     2888
8     2540
11    2392
10    2008
9     1968
12    1596
5     1448
3     1264
6     1180
1      792
7      572
0      256
Name: count, dtype: int64

In [18]:
# Create dummy variables without dropping the first category
long_df = pd.get_dummies(long_df, columns=['deh'], prefix='deh', drop_first=False)
long_df = pd.get_dummies(long_df, columns=['dms'], prefix='dms', drop_first=False)
long_df = pd.get_dummies(long_df, columns=['dcz'], prefix='dcz', drop_first=False)
long_df = pd.get_dummies(long_df, columns=['lindi'], prefix='lindi', drop_first=False)
long_df = pd.get_dummies(long_df, columns=['amrtn'], prefix='amrtn', drop_first=False)
long_df = pd.get_dummies(long_df, columns=['les'], prefix='les', drop_first=False)

# List of modes for each categorical variable
modes = {
    'deh': mode_deh,
    'dms': mode_dms,
    'dcz': mode_dcz,
    'lindi': mode_lindi,
    'amrtn': mode_amrtn,
    'les': mode_les
}

# Remove dummy columns corresponding to the mode
for var, mode in modes.items():
    dummy_col = f"{var}_{mode}"
    if dummy_col in long_df.columns:
        long_df = long_df.drop(columns=[dummy_col])


In [19]:
#copy
df = long_df.copy()


#mapping and filtration

def map_lhw(row):
    # Ensure the scenario prefix 'h' is included when constructing the column name
    scenario_prefix = 'h' if not row["scenario"].startswith('h') else ''
    scenario_column_name = f'lhw_{scenario_prefix}{row["scenario"]}'
    return row[scenario_column_name]

# Apply the corrected function
df['lhw_scenario'] = df.apply(map_lhw, axis=1)

# Print a sample to verify the column has been created correctly
print(df[['idperson', 'scenario', 'lhw_scenario']].head())
# Identify individuals with any negative 'ils_udb_yds' values
negative_c_ids = df[df['hyds'] <=  0]['idperson'].unique()
# Identify individuals with any negative 'ils_udb_yds' values
negative_l_ids = df[df['lhw_scenario'] >= 80 ]['idperson'].unique()
print(f"Number of households with negative leisure: {len(negative_l_ids)}")
print(f"Number of households with negative consumption: {len(negative_c_ids)}")



       idperson scenario  lhw_scenario
0      87060001       h0           0.0
11030  87060001       h2          44.0
16545  87060001       h3          57.0
5515   87060001       h1          14.0
1      87070001       h0           0.0
Number of households with negative leisure: 3
Number of households with negative consumption: 74


In [20]:
# Filter df to exclude all rows belonging to individuals identified in step 1 or step 2
df_filtered = df[~df['idperson'].isin(negative_c_ids) & ~df['idperson'].isin(negative_l_ids)]
# Verify the removal
print(f"Original dataframe size: {df.shape}")
print(f"Filtered dataframe size: {df_filtered.shape}")


Original dataframe size: (22060, 409)
Filtered dataframe size: (21752, 409)


In [21]:
# Ensure we are not modifying a copy of a slice
df_filtered = df_filtered.copy()

# Creating logs for translog and for quadratic
df_filtered.loc[:, 'consum'] = np.log(df_filtered['hyds'] + 1)
df_filtered.loc[:, 'logleis'] = np.log(80 - df_filtered['lhw_scenario'] + 1)
df_filtered.loc[:, 'logy'] = np.log(df_filtered['hyds'] + 1)
df_filtered.loc[:, 'logl'] = np.log(80 - df_filtered['lhw_scenario'] + 1)
df_filtered.loc[:, 'logy2'] = df_filtered['logy'] ** 2
df_filtered.loc[:, 'logl2'] = df_filtered['logl'] ** 2
df_filtered.loc[:, 'logyl'] = df_filtered['logy'] * df_filtered['logl']
df_filtered.loc[:, 'leis'] = 80 - df_filtered['lhw_scenario']
df_filtered.loc[:, 'leis2'] = df_filtered['leis'] ** 2
df_filtered.loc[:, 'hyds2'] = df_filtered['hyds'] ** 2
df_filtered.loc[:, 'dag2'] = df_filtered['dag'] ** 2

# Sorting the DataFrame by 'idperson'
df_filtered = df_filtered.sort_values(by='idperson')




In [22]:
# Make a copy of the filtered DataFrame
long_df = df_filtered.copy()

# List all columns that are scenario-specific
scenario_specific_columns = ['logy', 'logl', 'logy2', 'logl2', 'logyl', 'hyds','leis','leis2','hyds2']

# Identify all columns in the long_df that are not scenario-specific
non_scenario_columns = [col for col in long_df.columns if col not in scenario_specific_columns + ['scenario']]

# Transform to wide format for scenario-specific columns
wide_df = long_df.pivot(index='idperson', columns='scenario', values=scenario_specific_columns)
wide_df.columns = [f'{var}_{scen}' for var, scen in wide_df.columns]
wide_df.reset_index(inplace=True)

# Merge the wide format DataFrame with the non-scenario-specific columns
wide_df = wide_df.merge(long_df[non_scenario_columns].drop_duplicates(), on='idperson', how='left')

# Merge the 'original_scenario' to mark the chosen scenario
wide_df = wide_df.merge(filtered_df[['idperson', 'original_scenario']], on='idperson', how='left')

# Check for the duplicated columns and drop one of them
if 'original_scenario_x' in wide_df.columns and 'original_scenario_y' in wide_df.columns:
    wide_df = wide_df.drop(columns=['original_scenario_x'])
    wide_df = wide_df.rename(columns={'original_scenario_y': 'original_scenario'})
# Convert boolean columns to integers
for col in wide_df.select_dtypes(include='bool').columns:
    wide_df[col] = wide_df[col].astype(int)


## mapping
scenario_mapping = {'h0': 0, 'h1': 1, 'h2': 2, 'h3': 3}

wide_df['original_scenario'] = wide_df['original_scenario'].map(scenario_mapping)

wide_df.head()

,idperson,logy_h0,logy_h1,logy_h2,logy_h3,logl_h0,logl_h1,logl_h2,logl_h3,logy2_h0,logy2_h1,logy2_h2,logy2_h3,logl2_h0,logl2_h1,logl2_h2,logl2_h3,logyl_h0,logyl_h1,logyl_h2,logyl_h3,hyds_h0,hyds_h1,hyds_h2,hyds_h3,leis_h0,leis_h1,leis_h2,leis_h3,leis2_h0,leis2_h1,leis2_h2,leis2_h3,hyds2_h0,hyds2_h1,hyds2_h2,hyds2_h3,idhh,idmother,idfather,idpartner,idorighh,idorigperson,dag,dgn,dec,dwt,drgn2,ddi,dlg_s,ddilv,dmb,dsu01,drgur,drgmd,drgru,ddt,dsu00,dsu02,dncsy,dct,dehde,dey,drgn1,dsr,loc,loopcount_pens,liwft_s,lhw,lhwsr_s,lhwsesr_s,lunmy_s,lunmy,liwmy_s,liwwh,liwmy_a,lnu,lhwpv_a,liwmy02_a,lfs,lhw_a,lcb_a,lhwsr_a,lmc20,lhw20_a,lhw19_a,lhw18_a,lma20,lma19,lma18,lma,lmc,lcs,liwmy,liwftmy,liwptmy,lpemy,lse,liwmy_f,lhw_f,liwwh_f,lunmy_f,yem,yse,yemmc_s,yiy,yot,ypr,ypt,ypp,yemxm_s,yemmy,ysemy,yemmw_s,ysemw_s,ysemc_s,yempv_s,yivwg,yempv_a,ysv,yem_a,ydses_o,yptmp,ymwdt,ysepv_a,yemmy19_a,yemmy18_a,yemmy_a,yem20_a,yem19_a,yem18_a,yemmy20_a,yprrt,yemmwmy_s,ysemwmy_s,yds,poa00,pdi00,pdicm,pdinc,psuwd00,poacm,poanc,poaot,poa,pdiot,pdi,psuwdcm,psuot,psu,poanc00_s,poancna_s,poancrg_s,poanc_s,poacm_s,psuwdcm_s,pdimy,poamy,psumy,bsarg_s,bsa00_s,bwkmceemy_s,bunnc_s,bunct_s,bhl,bma,bwkmcee_s,bwkmcse_s,bwkmcsemy_s,bunct,bunctmy,bunctmy_s,bunctpc,bunnc,bunncmy,bed,bunncmy_s,bsa,bunmtmy_s,bch00_s,bchdi_s,bchbamtna_s,bchbaucna02_s,bchrg_s,bchucrg_s,bchmtrg_s,bchbarg_s,bchbaucrg_s,bchbamtrg_s,bchlgrg_s,bchlgucrg_s,bchlgmtrg_s,bunmt_s,bunot,bho,bunct02_s,bsa_s,bchbaucna_s,bunotmy,bhlot,bhl00,bmact_s,bpact_s,bmanc_s,bwkmcmy_a,bch,bch00,bchot,bchdi,bun,bfa,bunmy,xmp,xpp,xhcmomi,xhcmomc,xhcmo,xhcrt,xcc,xed00,xhl00,xhc,xhcot,tpr,tsceemy_s,tsctbee_s,tsceepi_s,tsceeui_s,tsceeot_s,tscee_s,tscerpi_s,tscerui_s,tscerot_s,tscer_s,tscsemy_s,tsctbse_s,tscsepi_s,tscsehl_s,tscseot_s,tscse_s,tsctbun_s,tscuner_s,tscunee_s,tsccterpi_s,tsccterui_s,tsccterot_s,tsccter_s,tscbeeepi_s,tscbeeeui_s,tscbeeeot_s,tscctsepi_s,tinty_s,tintaxp_s,tintbit_s,tinit_s,tintbjt_s,tinjt_s,tintp_s,tin_s,tinrg_s,tinna_s,tingt_s,tingtna_s,tingtrg_s,tinta00_s,tintach_s,tintadp_s,tintaee_s,tintb_s,tintbiy_s,tintc00_s,tintcmg_s,tintcrt_s,tintc_s,tintcrg_s,tintcbarg_s,tintcdbrg_s,tintclprg_s,tintcdprg_s,tintcoarg_s,tintcfarg_s,tintclgrg_s,tintcchrg_s,tintcccrg_s,tintcrtrg_s,tintceerg_s,tintchkrg_s,tintcwmrg_s,tintrch_s,tintrchlg_s,tintrchlp_s,twl,tin,tscee,tscse,tintrch,tad,tis,tscer,kfbcc,kivho,kfb,kfbmy,afc,amrrm,aco,aca,ils_earns,ils_origy,ils_pen,ils_origrepy,ils_bensim,ils_benmt,ils_bennt,ils_ben,ils_taxsim,ils_tax,ils_sicee,ils_sicer,ils_sicct,ils_sicse,ils_sicot,ils_sicdy,ils_dispy,ils_b1_boa,ils_b1_bsu,ils_b1_bdi,ils_b1_bun,ils_b1_bhl,ils_b1_bed,ils_b1_bsa,ils_b1_bcb,ils_b1_bfa,ils_b1_bho,ils_b2_bfaed,ils_b2_penhl,ils_b2_bsaho,ils_base_tin,ils_udb_yem,ils_udb_yse,ils_udb_ypp,ils_udb_ypr,ils_udb_yiy,ils_udb_ypt,ils_udb_yot,ils_udb_xmp,ils_udb_kfbcc,ils_udb_boa,ils_udb_bsu,ils_udb_bdi,ils_udb_bun,ils_udb_bhl,ils_udb_bed,ils_udb_bsa,ils_udb_bfa,ils_udb_bho,ils_udb_tpr,ils_udb_tis,ils_udb_yds,il_sic,il_tsctbeese,il_bch00,il_bsa_global,il_yse_net,il_bunnc,il_poanc,il_poacm,il_poacmds,il_psuwdcm,il_tinwkgr,il_tinwk,il_tinwk01,il_tinot,il_tinty,il_tintcit,il_tintbiit,il_tintbit,il_tintsit,il_tintcitrg,il_tintcmo,il_tintcjt,il_tintbijt,il_tintbjt,il_tintsjt,il_tintcjtrg,il_bsa00,il_bsarg_global,il_bsarg_11,il_bsarg_12,il_bsarg_13,il_bsarg_21,il_bsarg_22,il_bsarg_23,il_bsarg_24,il_bsarg_30,il_bsarg_41,il_bsarg_42,il_bsarg_43,il_bsarg_51,il_bsarg_52,il_bsarg_53,il_bsarg_61,il_bsarg_62,il_bsarg_63,il_bsarg_64,il_bsarg_70,lhw_h0,lhw_h1,lhw_h2,lhw_h3,yds_other,choice_made,leisure,deh_0,deh_1,deh_2,deh_3,deh_4,dms_3,dms_4,dms_5,dcz_2,dcz_3,lindi_0,lindi_1,lindi_3,lindi_4,lindi_5,lindi_6,lindi_7,lindi_8,lindi_9,lindi_10,lindi_11,lindi_12,amrtn_2,amrtn_3,amrtn_4,amrtn_6,les_2,les_7,lhw_scenario,consum,logleis,dag2,original_scenario
0,87060001,6.735067,6.936673,7.841831,8.059109,4.394449,4.204693,3.610918,3.178054,45.361130,48.117434,61.494312,64.949232,19.311183,17.67944,13.038728,10.100026,29.596910,29.1665

### Wide Format

In [23]:
# Filter the DataFrame to keep only the necessary columns
required_columns = [
    'logy_h0', 'logl_h0', 'logy2_h0', 'logl2_h0', 'logyl_h0',
    'logy_h1', 'logl_h1', 'logy2_h1', 'logl2_h1', 'logyl_h1',
    'logy_h2', 'logl_h2', 'logy2_h2', 'logl2_h2', 'logyl_h2',
    'logy_h3', 'logl_h3', 'logy2_h3', 'logl2_h3', 'logyl_h3',
    'original_scenario', 'dag', 'dag2', 'dgn', 'dncsy',
    'deh_1', 'deh_0','deh_3', 'deh_2',  'deh_4','dcz_2', 'dcz_3',
    'lindi_0','lindi_1', 'lindi_3','lindi_4', 'lindi_5', 'lindi_6','lindi_7', 'lindi_8','lindi_9','lindi_10','lindi_11','lindi_12',
    'les_2','les_7', 
    'amrtn_2', 'amrtn_3', 'amrtn_4',  'amrtn_6',
    'dms_3', 'dms_4' ,'dms_5'
]

# Filter the DataFrame
filtered_df = wide_df[required_columns]

# Ensure appropriate data types
filtered_df = filtered_df.astype({col: 'float32' for col in filtered_df.select_dtypes(include=[np.float64]).columns})

# Create the Biogeme database
database = db.Database('labour_supply', filtered_df)
globals().update(database.variables)


In [ ]:
filtered_df.to_csv('SpanishData.csv', index= False)

In [24]:
# Define variables for each scenario
logy_vars = {f'logy_h{i}': Variable(f'logy_h{i}') for i in range(4)}
logl_vars = {f'logl_h{i}': Variable(f'logl_h{i}') for i in range(4)}
logy2_vars = {f'logy2_h{i}': Variable(f'logy2_h{i}') for i in range(4)}
logl2_vars = {f'logl2_h{i}': Variable(f'logl2_h{i}') for i in range(4)}
logyl_vars = {f'logyl_h{i}': Variable(f'logyl_h{i}') for i in range(4)}
choice = Variable('original_scenario')

# Define heterogeneity variables
dag = Variable('dag')
dag2 = Variable('dag2')
dgn = Variable('dgn')
dncsy = Variable('dncsy')

# Define the parameters to be estimated with realistic boundaries
alpha = Beta('alpha', 41.9, 0, 100, 0)
beta = Beta('beta', 99, 0, 200, 0)
gamma_yy = Beta('gamma_yy', -1.01, -10, 0, 0)
gamma_ll = Beta('gamma_ll', -8.9, -20, 0, 0)
gamma_yl = Beta('gamma_yl', 0.019, -5, 5, 0)

# Define heterogeneity parameters for age and gender with boundaries
alpha_dag = Beta('alpha_dag', -0.0636, -10, 10, 0)
alpha_dag2 = Beta('alpha_dag2', 0.001107, -1, 1, 0)
alpha_dgn = Beta('alpha_dgn', 0.787, -10, 10, 0)
alpha_dncsy = Beta('alpha_dncsy', -2.61, -10, 10, 0)

beta_dag = Beta('beta_dag', -0.135, -10, 10, 0)
beta_dag2 = Beta('beta_dag2', 0.001558, -1, 1, 0)
beta_dgn = Beta('beta_dgn', -0.724, -5, 0, 0)
beta_dncsy = Beta('beta_dncsy', -2.62, -10, 0, 0)

# Create alpha and beta dictionaries for dummy variables with realistic boundaries
def create_alpha_beta_dict(levels, prefix):
    return {level: Beta(f'{prefix}_{level}', 0, -50, 50, 0) for level in levels}

deh_levels = [col for col in filtered_df.columns if 'deh_' in col]
dcz_levels = [col for col in filtered_df.columns if 'dcz_' in col]
lindi_levels = [col for col in filtered_df.columns if 'lindi_' in col]
les_levels = ['les_2', 'les_7']
amrtn_levels = [col for col in filtered_df.columns if 'amrtn_' in col]
dms_levels = [col for col in filtered_df.columns if 'dms_' in col]

alpha_deh = create_alpha_beta_dict(deh_levels, 'alpha')
beta_deh = create_alpha_beta_dict(deh_levels, 'beta')
alpha_dcz = create_alpha_beta_dict(dcz_levels, 'alpha')
beta_dcz = create_alpha_beta_dict(dcz_levels, 'beta')
alpha_lindi = create_alpha_beta_dict(lindi_levels, 'alpha')
beta_lindi = create_alpha_beta_dict(lindi_levels, 'beta')
alpha_les = create_alpha_beta_dict(les_levels, 'alpha')
beta_les = create_alpha_beta_dict(les_levels, 'beta')
alpha_amrtn = create_alpha_beta_dict(amrtn_levels, 'alpha')
beta_amrtn = create_alpha_beta_dict(amrtn_levels, 'beta')
alpha_dms = create_alpha_beta_dict(dms_levels, 'alpha')
beta_dms = create_alpha_beta_dict(dms_levels, 'beta')

# Combine all alpha and beta dictionaries
alpha_dict = {**alpha_deh, **alpha_dcz, **alpha_lindi, **alpha_les, **alpha_amrtn, **alpha_dms}
beta_dict = {**beta_deh, **beta_dcz, **beta_lindi, **beta_les, **beta_amrtn, **beta_dms}

# Precompute common terms
alpha_common_terms = alpha + alpha_dag * dag + alpha_dag2 * dag2 + alpha_dgn * dgn + alpha_dncsy * dncsy
beta_common_terms = beta + beta_dag * dag + beta_dag2 * dag2 + beta_dgn * dgn + beta_dncsy * dncsy

# Helper function to create utility
def create_utility(logy, logl, logy2, logl2, logyl, alpha_dict, beta_dict, alpha_common, beta_common):
    alpha_sum = alpha_common + sum(alpha_dict[level] * Variable(level) for level in alpha_dict)
    beta_sum = beta_common + sum(beta_dict[level] * Variable(level) for level in beta_dict)
    return alpha_sum * logy + beta_sum * logl + gamma_yy * logy2 + gamma_ll * logl2 + gamma_yl * logyl


In [26]:
# Generate the utility functions using the helper function
V = {i: create_utility(logy_vars[f'logy_h{i}'], logl_vars[f'logl_h{i}'], logy2_vars[f'logy2_h{i}'], logl2_vars[f'logl2_h{i}'], logyl_vars[f'logyl_h{i}'],
                       alpha_dict, beta_dict, alpha_common_terms, beta_common_terms) for i in range(4)}

# Define the availability of each choice (all choices are available)
av = {i: 1 for i in range(4)}

# Associate the utility functions with the choice situations
logprob = models.loglogit(V, av, choice)

# Create the Biogeme object
biogeme_model = bio.BIOGEME(database, logprob)
biogeme_model.modelName = 'optimized_model_with_bounds'
biogeme_model.choice = choice

# Adjust optimization settings for better convergence
biogeme_model.algorithmParameters = {'tolerance': 1e-6, 'maxIter': 2000}

# Estimate the parameters
results = biogeme_model.estimate()

# Print the estimated parameters
print(results.getEstimatedParameters())


It seems that the optimization algorithm did not converge. Therefore, the results may not correspond to the maximum likelihood estimator. Check the specification of the model, or the criteria for convergence of the algorithm.


                    Value  Active bound   Rob. Std err    Rob. t-test  \
alpha           17.593717           0.0   6.462833e-01   2.722292e+01   
alpha_amrtn_2   -0.903067           0.0  1.797693e+308 -5.023480e-309   
alpha_amrtn_3   -1.152147           0.0   4.837650e-01  -2.381626e+00   
alpha_amrtn_4   -2.004183           0.0   1.322018e-07  -1.516004e+07   
alpha_amrtn_6   -1.531450           0.0   5.868393e-01  -2.609659e+00   
alpha_dag       -0.065524           0.0   1.546037e-07  -4.238223e+05   
alpha_dag2       0.001131           0.0   8.292955e-08   1.363233e+04   
alpha_dcz_2     -1.511572           0.0   3.897605e-02  -3.878208e+01   
alpha_dcz_3     -1.336080           0.0   1.001502e+00  -1.334075e+00   
alpha_deh_0     -2.292734           0.0   6.133349e-01  -3.738145e+00   
alpha_deh_1     -1.294948           0.0  1.797693e+308 -7.203385e-309   
alpha_deh_2     -1.343342           0.0   1.515418e-01  -8.864503e+00   
alpha_deh_3     -0.651510           0.0  1.797693e+

In [ ]:

# Estimate the parameters
results = biogeme_model.estimate()

# Print the estimated parameters
print(results.getEstimatedParameters())


In [13]:
# Calculate prediction accuracy
# Simulate probabilities for each alternative
simulate = {
    'Prob0': models.logit(V, av, 0),
    'Prob1': models.logit(V, av, 1),
    'Prob2': models.logit(V, av, 2),
    'Prob3': models.logit(V, av, 3)
}

biogeme_sim = bio.BIOGEME(database, simulate)
simulated_values = biogeme_sim.simulate(results.getBetaValues())

# Determine the predicted choice by selecting the alternative with the highest probability
simulated_values['Predicted'] = simulated_values[['Prob0', 'Prob1', 'Prob2', 'Prob3']].idxmax(axis=1).apply(lambda x: int(x[-1]))

# Merge predictions with actual choices
simulated_values = simulated_values.merge(wide_df[['idperson', 'original_scenario']], left_index=True, right_index=True)

# Calculate the percentage of correctly predicted choices
correct_predictions = simulated_values[simulated_values['Predicted'] == simulated_values['original_scenario']]
percent_correct = len(correct_predictions) / len(simulated_values) * 100

print(f'Percent correctly predicted: {percent_correct:.2f}%')


Percent correctly predicted: 80.93%


In [31]:
print(results.getEstimatedParameters(onlyRobust=False))

                    Value  Active bound    Std err     t-test       p-value  \
alpha           17.593717           0.0   1.892970   9.294239  0.000000e+00   
alpha_amrtn_2   -0.903067           0.0   0.194676  -4.638823  3.503989e-06   
alpha_amrtn_3   -1.152147           0.0   0.210982  -5.460874  4.737955e-08   
alpha_amrtn_4   -2.004183           0.0   0.372607  -5.378810  7.497984e-08   
alpha_amrtn_6   -1.531450           0.0   0.264541  -5.789089  7.076940e-09   
alpha_dag       -0.065524           0.0   0.054838  -1.194870  2.321378e-01   
alpha_dag2       0.001131           0.0   0.000610   1.852785  6.391319e-02   
alpha_dcz_2     -1.511572           0.0   0.298173  -5.069439  3.989893e-07   
alpha_dcz_3     -1.336080           0.0   0.279147  -4.786296  1.698874e-06   
alpha_deh_0     -2.292734           0.0   0.510983  -4.486907  7.226468e-06   
alpha_deh_1     -1.294948           0.0   0.333209  -3.886300  1.017835e-04   
alpha_deh_2     -1.343342           0.0   0.207021  

In [29]:
results.getGeneralStatistics()

{'Number of estimated parameters': GeneralStatistic(value=69, format=''),
 'Number of free parameters': GeneralStatistic(value=68, format=''),
 'Sample size': GeneralStatistic(value=21752, format=''),
 'Excluded observations': GeneralStatistic(value=0, format=''),
 'Init log likelihood': GeneralStatistic(value=-95411.02884190489, format='.7g'),
 'Final log likelihood': GeneralStatistic(value=-11649.578494082052, format='.7g'),
 'Likelihood ratio test for the init. model': GeneralStatistic(value=167522.90069564566, format='.7g'),
 'Rho-square for the init. model': GeneralStatistic(value=0.8779011332810875, format='.3g'),
 'Rho-square-bar for the init. model': GeneralStatistic(value=0.8771779464457969, format='.3g'),
 'Akaike Information Criterion': GeneralStatistic(value=23437.156988164104, format='.7g'),
 'Bayesian Information Criterion': GeneralStatistic(value=23988.291796219655, format='.7g'),
 'Final gradient norm': GeneralStatistic(value=0.6001566480186642, format='.4E'),
 'Nbr of 

In [17]:
# Generate the utility functions using the helper function
V = {i: create_utility(logy_vars[f'logy_h{i}'], logl_vars[f'logl_h{i}'], logy2_vars[f'logy2_h{i}'], logl2_vars[f'logl2_h{i}'], logyl_vars[f'logyl_h{i}'],
                       alpha_dict, beta_dict, alpha_common_terms, beta_common_terms) for i in range(4)}

# Define the availability of each choice (all choices are available)
av = {i: 1 for i in range(4)}

# Associate the utility functions with the choice situations
logprob = models.loglogit(V, av, choice)

# Create the Biogeme object
biogeme_model = bio.BIOGEME(database, logprob)
biogeme_model.modelName = 'optimized_model_with_bounds2'
biogeme_model.choice = choice

# Adjust optimization settings for better convergence
biogeme_model.algorithmParameters = {'tolerance': 1e-6, 'maxIter': 2000}


# Enable parallel processing if supported
biogeme_model.saveIterations = True
biogeme_model.numberOfThreads = 12 # Adjust based on your CPU capabilities


In [18]:
# Estimate the parameters
results2 = biogeme_model.estimate()

# Print the estimated parameters
print(results2.getEstimatedParameters())



KeyboardInterrupt



In [23]:
cProfile.run('biogeme_model.estimate()')

It seems that the optimization algorithm did not converge. Therefore, the results may not correspond to the maximum likelihood estimator. Check the specification of the model, or the criteria for convergence of the algorithm.


         5873969 function calls (5755980 primitive calls) in 19937.632 seconds

   Ordered by: standard name

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
     4830    0.003    0.000    0.037    0.000 <__array_function__ internals>:177(all)
      103    0.000    0.000    0.004    0.000 <__array_function__ internals>:177(amax)
      100    0.000    0.000    0.001    0.000 <__array_function__ internals>:177(amin)
    19432    0.018    0.000    0.167    0.000 <__array_function__ internals>:177(any)
        1    0.000    0.000    0.000    0.000 <__array_function__ internals>:177(argmax)
        1    0.000    0.000    0.000    0.000 <__array_function__ internals>:177(argmin)
     4830    0.003    0.000    0.018    0.000 <__array_function__ internals>:177(atleast_1d)
     4830    0.004    0.000    0.122    0.000 <__array_function__ internals>:177(broadcast_arrays)
      203    0.001    0.000    0.014    0.000 <__array_function__ internals>:177(clip)
     4826    0.

In [ ]:
# Define the coefficients and heterogeneity parameters
alpha = 17.568297 
beta = 69.308100
gamma_yy = -1.011667 
gamma_ll = -8.918268
gamma_yl = 0.003867

# Heterogeneity parameters
alpha_amrtn_2 = -0.901873
alpha_amrtn_3 = -1.153096 
alpha_amrtn_4 = -2.002610   
alpha_amrtn_6 = -1.529547 
alpha_dag = -0.064722 
alpha_dag2 = 0.001121
alpha_dcz_2 = -1.507303 
alpha_dcz_3 = -1.333377 
alpha_dgn = 0.788530 
alpha_deh_0 = -2.291884
alpha_deh_1 = -1.291191
alpha_deh_2 = -1.340360
alpha_deh_3 = -0.649827
alpha_deh_4 = 2.524951
alpha_dms_3 = -0.380716 
alpha_dms_4 = -0.312
alpha_dms_5 = 0.211971
alpha_dncsy = -2.591300
alpha_les_2 = -4.116105      
alpha_les_7 = 9.074582
alpha_lindi_0 = 55.990346
alpha_lindi_1 = -1.647418 
alpha_lindi_10 = -0.272774 
alpha_lindi_11 = 0.048701
alpha_lindi_12 = -1.805131 
alpha_lindi_3 = -0.845782
alpha_lindi_4 = 0.559927
alpha_lindi_5 = -1.910248
alpha_lindi_6 = -2.608789
alpha_lindi_7 = 1.022106
alpha_lindi_8 = -0.022022	
alpha_lindi_9 = 0.192925

beta_amrtn_2 = 0.109422 
beta_amrtn_3 = -0.227860
beta_amrtn_4 = -1.145363 
beta_amrtn_6 = -0.029801
beta_dag = -0.138533 
beta_dag2 = 0.001593
beta_dcz_2 = -1.431550 
beta_dcz_3 = -0.235669 
beta_deh_0 = 0.357007
beta_deh_1 = 0.442505
beta_deh_2 = -0.207603	
beta_deh_3 = 0.567477
beta_deh_4 = 2.990132	
beta_dgn = -0.725227  
beta_dms_3 = 0.342493
beta_dms_4 = -0.283816 
beta_dms_5 = 0.306725 
beta_dncsy = -2.604926 
beta_les_2 = -6.044054 
beta_les_7 = 2.184340 
beta_lindi_0 = 59.663166
beta_lindi_1 = -1.346355	
beta_lindi_10 = 0.743129	
beta_lindi_11 = -0.099516	
beta_lindi_12 = 0.000875
beta_lindi_3 = -0.564887
beta_lindi_4 = 0.333134
beta_lindi_5 = -1.870053
beta_lindi_6 = -2.618889
beta_lindi_7 = 1.715977 
beta_lindi_8 = 0.481687
beta_lindi_9 = 0.590077 

In [80]:
# Filter the DataFrame to keep only the necessary columns
required_columns = [
    'logy_h0', 'logl_h0', 'logy2_h0', 'logl2_h0', 'logyl_h0',
    'logy_h1', 'logl_h1', 'logy2_h1', 'logl2_h1', 'logyl_h1',
    'logy_h2', 'logl_h2', 'logy2_h2', 'logl2_h2', 'logyl_h2',
    'logy_h3', 'logl_h3', 'logy2_h3', 'logl2_h3', 'logyl_h3',
    'original_scenario', 'dag', 'dag2', 'dgn', 'dncsy',
    'deh_1', 'deh_0','deh_3', 'deh_2',  'deh_4','dcz_2', 'dcz_3',
    'lindi_0','lindi_1', 'lindi_3','lindi_4', 'lindi_5', 'lindi_6','lindi_7', 'lindi_8','lindi_9','lindi_10','lindi_11','lindi_12',
    'les_2','les_7', 'choice_made',
    'amrtn_2', 'amrtn_3', 'amrtn_4',  'amrtn_6',
    'dms_3', 'dms_4' ,'dms_5'
]

# Filter the DataFrame
filtered_df = wide_df[required_columns]

# Ensure appropriate data types
filtered_df = filtered_df.astype({col: 'float32' for col in filtered_df.select_dtypes(include=[np.float64]).columns})

# Create the Biogeme database
database = db.Database('labour_supply', filtered_df)
globals().update(database.variables)


In [81]:
# Define variables for each scenario
logy_vars = {f'logy_h{i}': Variable(f'logy_h{i}') for i in range(4)}
logl_vars = {f'logl_h{i}': Variable(f'logl_h{i}') for i in range(4)}
logy2_vars = {f'logy2_h{i}': Variable(f'logy2_h{i}') for i in range(4)}
logl2_vars = {f'logl2_h{i}': Variable(f'logl2_h{i}') for i in range(4)}
logyl_vars = {f'logyl_h{i}': Variable(f'logyl_h{i}') for i in range(4)}
choice = Variable('original_scenario')

# Define heterogeneity variables
dag = Variable('dag')
dag2 = Variable('dag2')
dgn = Variable('dgn')
dncsy = Variable('dncsy')

# Define the parameters to be estimated with realistic boundaries
alpha = Beta('alpha', 41.9, 0, 100, 0)
beta = Beta('beta', 99, 0, 200, 0)
gamma_yy = Beta('gamma_yy', -1.01, -10, 0, 0)
gamma_ll = Beta('gamma_ll', -8.9, -20, 0, 0)
gamma_yl = Beta('gamma_yl', 0.019, -5, 5, 0)

# Define heterogeneity parameters for age and gender with boundaries
alpha_dag = Beta('alpha_dag', -0.0636, -10, 10, 0)
alpha_dag2 = Beta('alpha_dag2', 0.001107, -1, 1, 0)
alpha_dgn = Beta('alpha_dgn', 0.787, -10, 10, 0)
alpha_dncsy = Beta('alpha_dncsy', -2.61, -10, 10, 0)

beta_dag = Beta('beta_dag', -0.135, -10, 10, 0)
beta_dag2 = Beta('beta_dag2', 0.001558, -1, 1, 0)
beta_dgn = Beta('beta_dgn', -0.724, -5, 0, 0)
beta_dncsy = Beta('beta_dncsy', -2.62, -10, 0, 0)

# Create alpha and beta dictionaries for dummy variables with realistic boundaries
def create_alpha_beta_dict(levels, prefix):
    return {level: Beta(f'{prefix}_{level}', 0, -50, 50, 0) for level in levels}

deh_levels = [col for col in filtered_df.columns if 'deh_' in col]
dcz_levels = [col for col in filtered_df.columns if 'dcz_' in col]
lindi_levels = [col for col in filtered_df.columns if 'lindi_' in col]
les_levels = ['les_2', 'les_7']
amrtn_levels = [col for col in filtered_df.columns if 'amrtn_' in col]
dms_levels = [col for col in filtered_df.columns if 'dms_' in col]

alpha_deh = create_alpha_beta_dict(deh_levels, 'alpha')
beta_deh = create_alpha_beta_dict(deh_levels, 'beta')
alpha_dcz = create_alpha_beta_dict(dcz_levels, 'alpha')
beta_dcz = create_alpha_beta_dict(dcz_levels, 'beta')
alpha_lindi = create_alpha_beta_dict(lindi_levels, 'alpha')
beta_lindi = create_alpha_beta_dict(lindi_levels, 'beta')
alpha_les = create_alpha_beta_dict(les_levels, 'alpha')
beta_les = create_alpha_beta_dict(les_levels, 'beta')
alpha_amrtn = create_alpha_beta_dict(amrtn_levels, 'alpha')
beta_amrtn = create_alpha_beta_dict(amrtn_levels, 'beta')
alpha_dms = create_alpha_beta_dict(dms_levels, 'alpha')
beta_dms = create_alpha_beta_dict(dms_levels, 'beta')

# Combine all alpha and beta dictionaries
alpha_dict = {**alpha_deh, **alpha_dcz, **alpha_lindi, **alpha_les, **alpha_amrtn, **alpha_dms}
beta_dict = {**beta_deh, **beta_dcz, **beta_lindi, **beta_les, **beta_amrtn, **beta_dms}

# Precompute common terms
alpha_common_terms = alpha + alpha_dag * dag + alpha_dag2 * dag2 + alpha_dgn * dgn + alpha_dncsy * dncsy
beta_common_terms = beta + beta_dag * dag + beta_dag2 * dag2 + beta_dgn * dgn + beta_dncsy * dncsy

# Helper function to create utility
def create_utility(logy, logl, logy2, logl2, logyl, alpha_dict, beta_dict, alpha_common, beta_common):
    alpha_sum = alpha_common + sum(alpha_dict[level] * Variable(level) for level in alpha_dict)
    beta_sum = beta_common + sum(beta_dict[level] * Variable(level) for level in beta_dict)
    return alpha_sum * logy + beta_sum * logl + gamma_yy * logy2 + gamma_ll * logl2 + gamma_yl * logyl

# Generate the utility functions using the helper function
V = {i: create_utility(logy_vars[f'logy_h{i}'], logl_vars[f'logl_h{i}'], logy2_vars[f'logy2_h{i}'], logl2_vars[f'logl2_h{i}'], logyl_vars[f'logyl_h{i}'],
                       alpha_dict, beta_dict, alpha_common_terms, beta_common_terms) for i in range(4)}

# Define the availability of each choice (all choices are available)
av = {i: 1 for i in range(4)}

# Associate the utility functions with the choice situations
logprob = models.loglogit(V, av, choice)

# Create the Biogeme object
biogeme_model = bio.BIOGEME(database, logprob)
biogeme_model.modelName = 'optimized_model_with_bounds'
biogeme_model.choice = choice


In [12]:
# Provided parameters (make sure these are aligned with your actual parameters)
parameters = np.array([
    17.593717, -0.903067, -1.152147, -2.004183, -1.531450, -0.065524, 0.001131,
    -1.511572, -1.336080, -2.292734, -1.294948, -1.343342, -0.651510, 2.528377,
    0.788545, 0.364318, -0.383762, 0.217083, -2.606557, -4.114244, 7.409098,
    46.734577, -1.649985, -0.275421, 0.045918, -1.809257, -0.842629, 0.558627,
    -1.909345, -2.608568, 1.014708, -0.030188, 0.191784, 69.266314, 0.109090,
    -0.226330, -1.147244, -0.029463, -0.138374, 0.001593, -1.431761, -0.236359,
    0.356880, 0.437164, -0.208159, 0.567141, 2.990752, -0.724239, 0.339303,
    -0.286117, 0.308062, -2.617150, -6.041479, 2.197993, 50.000000, -1.350432,
    0.744651, -0.100148, -0.001720, -0.564073, 0.333098, -1.872319, -2.617936,
    1.711621, 0.476389, 0.588318, -8.913998, 0.005028, -1.012031
])

def profile_function():
    lp = LineProfiler()
    # Ensure the function name is correct and exists in the biogeme module
    lp.add_function(biogeme_model.calculateLikelihoodAndDerivatives)
    lp_wrapper = lp(biogeme_model.calculateLikelihoodAndDerivatives)
    # Provide the actual arguments required by your function
    scaled = False
    hessian = False
    bhhh = False
    lp_wrapper(parameters, scaled, hessian, bhhh)
    lp.print_stats()

# Call the profiling function
profile_function()


Timer unit: 1e-07 s

Total time: 51.0275 s
File: C:\Users\hisham\AppData\Roaming\Python\Python311\site-packages\biogeme\biogeme.py
Function: calculateLikelihoodAndDerivatives at line 1025

Line #      Hits         Time  Per Hit   % Time  Line Contents
  1025                                               def calculateLikelihoodAndDerivatives(
  1026                                                   self, x, scaled, hessian=False, bhhh=False, batch=None
  1027                                               ):
  1028                                                   """Calculate the value of the log likelihood function
  1029                                                   and its derivatives.
  1030                                           
  1031                                                   :param x: vector of values for the parameters.
  1032                                                   :type x: list(float)
  1033                                           
  1034           

In [ ]:

# Adjust optimization settings for better convergence
biogeme_model.algorithmParameters = {'tolerance': 1e-6, 'maxIter': 2000}

# Estimate the parameters
results = biogeme_model.estimate()

# Print the estimated parameters
print(results.getEstimatedParameters())


## Long Format

In [52]:
# Filter the DataFrame to keep only the necessary columns
required_columns = [
    'logy', 'logl', 'logy2', 'logl2', 'logyl','scenario','idperson',
    'original_scenario', 'dag', 'dag2', 'dgn', 'dncsy', 
    'deh_1', 'deh_0','deh_3', 'deh_2',  'deh_4','dcz_2', 'dcz_3',
    'lindi_0','lindi_1', 'lindi_3','lindi_4', 'lindi_5', 'lindi_6','lindi_7', 'lindi_8','lindi_9','lindi_10','lindi_11','lindi_12',
    'les_2','les_7', 'choice_made',
    'amrtn_2', 'amrtn_3', 'amrtn_4',  'amrtn_6',
    'dms_3', 'dms_4' ,'dms_5'
]


# Filter the DataFrame
filtered_df = df_filtered[required_columns]

# Ensure 'choice_made' is correctly set
filtered_df.loc[:, 'choice_made'] = (filtered_df['original_scenario'] == filtered_df['scenario']).astype(int)

# Ensure appropriate data types for other columns if needed
filtered_df = filtered_df.astype({col: 'float32' for col in filtered_df.select_dtypes(include=[np.float64]).columns})

# Create the Biogeme database
database = db.Database('labour_supply', filtered_df)
globals().update(database.variables)



In [53]:
filtered_df.describe()

,logy,logl,logy2,logl2,logyl,scenario,idperson,original_scenario,dag,dag2,dgn,dncsy,deh_1,deh_0,deh_3,deh_2,deh_4,dcz_2,dcz_3,lindi_0,lindi_1,lindi_3,lindi_4,lindi_5,lindi_6,lindi_7,lindi_8,lindi_9,lindi_10,lindi_11,lindi_12,les_2,les_7,choice_made,amrtn_2,amrtn_3,amrtn_4,amrtn_6,dms_3,dms_4,dms_5
count,21752.000000,21752.000000,21752.000000,21752.000000,21752.000000,21752.00000,2.175200e+04,21752.000000,21752.000000,21752.000000,21752.000000,21752.000000,21752.000000,21752.000000,21752.000000,21752.000000,21752.000000,21752.000000,21752.000000,21752.000000,21752.000000,21752.000000,21752.000000,21752.000000,21752.000000,21752.000000,21752.000000,21752.000000,21752.000000,21752.000000,21752.000000,21752.000000,21752.000000,21752.000000,21752.000000,21752.000000,21752.000000,21752.000000,21752.000000,21752.000000,21752.000000
mean,7.440955,3.885083,56.012913,15.334471,28.695562,2.50000,4.282730e+08,2.978117,45.329901,2163.448694,0.486208,0.004229,0.051673,0.010666,0.252850,0.214049,0.003678,0.029790,0.043950,0.011769,0.036227,0.058110,0.131666,0.065649,0.053696,0.025193,0.115300,0.087716,0.091394,0.107944,0.073005,0.103531,0.015999,0.250000,0.335969,0.218095,0.029790,0.071718,0.057926,0.235748,0.051673
std,0.803203,0.490523,11.614237,3.685370,3.438837,1.11806,1.524512e+08,0.501278,10.423713,934.856971,0.499821,0.064898,0.221372,0.102725,0.434656,0.410171,0.060535,0.170012,0.204989,0.107847,0.186858,0.233956,0.338135,0.247673,0.225422,0.156715,0.319391,0.282888,0.288175,0.310317,0.260150,0.304658,0.125472,0.433023,0.472339,0.412962,0.170012,0.258026,0.233608,0.424475,0.221372
min,0.815365,1.098612,0.664820,1.206949,3.583079,1.00000,8.706000e+07,1.000000,17.000000,289.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,6.885791,3.488815,47.414115,12.172005,26.416073,1.75000,2.921700e+08,3.000000,38.000000,1444.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,7.566984,4.016342,57.259241,16.131088,28.286740,2.50000,4.839800e+08,3.000000,46.000000,2116.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
75%,8.025714,4.385132,64.412075,19.229414,31.102362,3.25000,5.565100e+08,3.000000,54.000000,2916.000000,1.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.250000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
max,9.927689,4.394449,98.558998,19.311184,41.805153,4.00000,6.201200e+08,4.000000,65.000000,4225.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000


In [54]:
# Define variables directly from the long format DataFrame
logy = Variable('logy')
logl = Variable('logl')
logy2 = Variable('logy2')
logl2 = Variable('logl2')
logyl = Variable('logyl')
choice = Variable('choice_made')

# Define heterogeneity variables (these don't change)
dag = Variable('dag')
dag2 = Variable('dag2')
dgn = Variable('dgn')
dncsy = Variable('dncsy')

# Define the parameters to be estimated
alpha = Beta('alpha', 41.9, 0, 100, 0)
beta = Beta('beta', 99, 0, 200, 0)
gamma_yy = Beta('gamma_yy', -1.01, -10, 0, 0)
gamma_ll = Beta('gamma_ll', -8.9, -20, 0, 0)
gamma_yl = Beta('gamma_yl', 0.019, -5, 5, 0)

# Define heterogeneity parameters for age and gender
alpha_dag = Beta('alpha_dag', -0.0636, -10, 10, 0)
alpha_dag2 = Beta('alpha_dag2', 0.001107, -1, 1, 0)
alpha_dgn = Beta('alpha_dgn', 0.787, -10, 10, 0)
alpha_dncsy = Beta('alpha_dncsy', -2.61, -10, 10, 0)

beta_dag = Beta('beta_dag', -0.135, -10, 10, 0)
beta_dag2 = Beta('beta_dag2', 0.001558, -1, 1, 0)
beta_dgn = Beta('beta_dgn', -0.724, -5, 0, 0)
beta_dncsy = Beta('beta_dncsy', -2.62, -10, 0, 0)

# Create alpha and beta dictionaries for dummy variables
def create_alpha_beta_dict(levels, prefix):
    return {level: Beta(f'{prefix}_{level}', 0, -50, 50, 0) for level in levels}

deh_levels = [col for col in filtered_df.columns if 'deh_' in col]
dcz_levels = [col for col in filtered_df.columns if 'dcz_' in col]
lindi_levels = [col for col in filtered_df.columns if 'lindi_' in col]
les_levels = ['les_2', 'les_7']
amrtn_levels = [col for col in filtered_df.columns if 'amrtn_' in col]
dms_levels = [col for col in filtered_df.columns if 'dms_' in col]

alpha_deh = create_alpha_beta_dict(deh_levels, 'alpha')
beta_deh = create_alpha_beta_dict(deh_levels, 'beta')
alpha_dcz = create_alpha_beta_dict(dcz_levels, 'alpha')
beta_dcz = create_alpha_beta_dict(dcz_levels, 'beta')
alpha_lindi = create_alpha_beta_dict(lindi_levels, 'alpha')
beta_lindi = create_alpha_beta_dict(lindi_levels, 'beta')
alpha_les = create_alpha_beta_dict(les_levels, 'alpha')
beta_les = create_alpha_beta_dict(les_levels, 'beta')
alpha_amrtn = create_alpha_beta_dict(amrtn_levels, 'alpha')
beta_amrtn = create_alpha_beta_dict(amrtn_levels, 'beta')
alpha_dms = create_alpha_beta_dict(dms_levels, 'alpha')
beta_dms = create_alpha_beta_dict(dms_levels, 'beta')

# Combine all alpha and beta dictionaries
alpha_dict = {**alpha_deh, **alpha_dcz, **alpha_lindi, **alpha_les, **alpha_amrtn, **alpha_dms}
beta_dict = {**beta_deh, **beta_dcz, **beta_lindi, **beta_les, **beta_amrtn, **beta_dms}

# Precompute common terms
alpha_common_terms = alpha + alpha_dag * dag + alpha_dag2 * dag2 + alpha_dgn * dgn + alpha_dncsy * dncsy
beta_common_terms = beta + beta_dag * dag + beta_dag2 * dag2 + beta_dgn * dgn + beta_dncsy * dncsy

# Define the utility function
alpha_sum = alpha_common_terms + sum(alpha_dict[level] * Variable(level) for level in alpha_dict)
beta_sum = beta_common_terms + sum(beta_dict[level] * Variable(level) for level in beta_dict)

V = alpha_sum * logy + beta_sum * logl + gamma_yy * logy2 + gamma_ll * logl2 + gamma_yl * logyl

# Define availability of each choice (all choices are available)
av = {1: 1, 2: 1, 3: 1, 4: 1}

# Associate the utility function with the choice situations
logprob = models.loglogit({1: V, 2: V, 3: V, 4: V}, av, choice)


In [55]:
# Create the Biogeme object
biogeme_model = bio.BIOGEME(database, logprob)
biogeme_model.modelName = 'long_optimized_model_with_bounds'
biogeme_model.choice = choice

# Adjust optimization settings for better convergence
biogeme_model.algorithmParameters = {'tolerance': 1e-6, 'maxIter': 2000}

# Estimate the parameters
resultslong = biogeme_model.estimate()

# Print the estimated parameters
print(resultslong.getEstimatedParameters())

BiogemeError: Chosen alternative 0.0 does not appear in availability dict: dict_keys([1, 2, 3, 4])